# KR1 — Plan Freeze Rate (Estabilidade do Planejamento)

**Objetivo**: Medir que % do plano original de cada ciclo sobreviveu sem alteração interna da Insider.

**Fórmula**: `KR1 = 1 - (Σ volume_alterado_interno / Σ volume_original)`

**Reason Codes Internos**: INT_DATE (mudança de dt_planned), INT_CANCEL (cancelamento In Season), INT_GRADE (mudança de grade)

**Reason Codes Externos** (acompanhamento): EXT_CANCEL (cancelamento por outro motivo), EXT_DATE_REV (fornecedor mudou dt_reviewed)

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import datetime


In [2]:
from google.cloud import bigquery

client = bigquery.Client(project="insider-data-lake")

today = pd.to_datetime("today").strftime("%Y%m%d")
sql_path = '../plan_freeze_rate/sql/'
output_path = '../../outputs/'
CICLOS_EXCLUIDOS = ['C012026']
WATERFALL_CYCLE_TYPE = 'Base'  # 'Base' | 'Extra' | None
WATERFALL_MES_ESCOLHIDO = '2026-06'
colors_cycle = {'Base': '#1f77b4', 'Extra': '#ff7f0e'}


/Users/insider/LA_Coding_Projects/.venv/lib/python3.13/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


# 1. Carregamento dos Dados


In [3]:
def query_to_dataframe(query):
    """Executa query no BigQuery e retorna DataFrame."""
    query_job = client.query(query)
    results = query_job.result()
    return results.to_dataframe()


def read_sql_file(file_path):
    """Lê arquivo SQL e retorna string."""
    with open(file_path, 'r') as file:
        return file.read()


def convert_sql_to_df(file_path, **params):
    """Lê SQL, substitui parâmetros e executa."""
    sql = read_sql_file(file_path)
    if params:
        sql = sql.format(**params)
    return query_to_dataframe(sql)


def load_plano(sql_path):
    df_plano = convert_sql_to_df(sql_path + 'plano_vs_atual.sql')
    print(f"Plano vs Atual: {df_plano.shape[0]:,} linhas (OP-SKU), {df_plano['cycle_name'].nunique()} ciclos")
    return df_plano


def load_frequencia(sql_path):
    df_freq = convert_sql_to_df(sql_path + 'frequencia_revisoes.sql')
    print(f"Frequência de revisões: {df_freq.shape[0]:,} linhas (OP-SKU)")
    return df_freq


def build_kr1_coorte(df_plano, df_freq, ciclos_excluidos):
    mask_excluir = df_plano['cycle_name'].isin(ciclos_excluidos)
    df_plano = df_plano[~mask_excluir].copy()
    df_freq = df_freq[~df_freq['cycle_name'].isin(ciclos_excluidos)].copy()
    print(f"Ciclos excluídos: {ciclos_excluidos}")
    print(f"df_plano após exclusão: {df_plano.shape[0]:,} linhas, {df_plano['cycle_name'].nunique()} ciclos")

    kr1_coorte = df_plano.groupby(['cycle_name', 'cycle_type']).agg(
        vol_original=('baseline_planned_qty', 'sum'),
        vol_int_date=('baseline_planned_qty', lambda x: x[df_plano.loc[x.index, 'is_int_date']].sum()),
        vol_int_cancel=('baseline_planned_qty', lambda x: x[df_plano.loc[x.index, 'is_int_cancel']].sum()),
        vol_int_grade=('baseline_planned_qty', lambda x: x[df_plano.loc[x.index, 'is_int_grade']].sum()),
        vol_int_any=('baseline_planned_qty', lambda x: x[df_plano.loc[x.index, 'is_int_any']].sum()),
        vol_ext_cancel=('baseline_planned_qty', lambda x: x[df_plano.loc[x.index, 'is_ext_cancel']].sum()),
        vol_ext_date_rev=('baseline_planned_qty', lambda x: x[df_plano.loc[x.index, 'is_ext_date_rev']].sum()),
        vol_ext_any=('baseline_planned_qty', lambda x: x[df_plano.loc[x.index, 'is_ext_any']].sum()),
        n_ops=('op_code', 'nunique'),
        n_skus=('product_sku', 'nunique'),
        baseline_date=('baseline_date', 'first'),
    ).reset_index()

    kr1_coorte['kr1'] = 1 - (kr1_coorte['vol_int_any'] / kr1_coorte['vol_original'])
    kr1_coorte['kr1_pct'] = (kr1_coorte['kr1'] * 100).round(1)

    for rc in ['int_date', 'int_cancel', 'int_grade']:
        kr1_coorte[f'pct_{rc}'] = (kr1_coorte[f'vol_{rc}'] / kr1_coorte['vol_original'] * 100).round(1)

    mes_alvo = df_plano.groupby('cycle_name').apply(
        lambda g: g.groupby(pd.to_datetime(g['baseline_dt_planned']).dt.to_period('M'))['baseline_planned_qty']
        .sum().idxmax()
    ).reset_index()
    mes_alvo.columns = ['cycle_name', 'mes_alvo']

    kr1_coorte = kr1_coorte.merge(mes_alvo, on='cycle_name', how='left')
    kr1_coorte = kr1_coorte[kr1_coorte['mes_alvo'] >= pd.Period('2025-11', freq='M')]
    print(f"Coortes após filtro (mes_alvo >= 2025-11): {len(kr1_coorte)}")

    ciclos_validos = kr1_coorte['cycle_name'].unique()
    df_plano = df_plano[df_plano['cycle_name'].isin(ciclos_validos)]
    df_freq = df_freq[df_freq['cycle_name'].isin(ciclos_validos)]
    print(f"df_plano filtrado: {df_plano.shape[0]:,} linhas")
    print(f"df_freq filtrado: {df_freq.shape[0]:,} linhas")

    kr1_coorte = kr1_coorte.sort_values(['mes_alvo', 'vol_original'], ascending=[True, False])

    print(f"\nTotal de coortes: {len(kr1_coorte)}")
    print(f"  Base: {(kr1_coorte['cycle_type'] == 'Base').sum()}")
    print(f"  Extra: {(kr1_coorte['cycle_type'] == 'Extra').sum()}")

    return df_plano, df_freq, kr1_coorte, ciclos_validos


def build_kr1_tipo(kr1_coorte):
    mes_atual = pd.Period(pd.Timestamp.today(), freq='M')
    kr1_coorte_okr = kr1_coorte[kr1_coorte['mes_alvo'] >= mes_atual]
    print(f"Coortes no OKR ativo (mes_alvo >= {mes_atual}): {len(kr1_coorte_okr)}")

    kr1_tipo = kr1_coorte_okr.groupby('cycle_type').agg(
        vol_original=('vol_original', 'sum'),
        vol_int_any=('vol_int_any', 'sum'),
        vol_ext_any=('vol_ext_any', 'sum'),
    ).reset_index()
    kr1_tipo['kr1'] = 1 - (kr1_tipo['vol_int_any'] / kr1_tipo['vol_original'])
    kr1_tipo['kr1_pct'] = (kr1_tipo['kr1'] * 100).round(1)
    kr1_tipo['meta'] = kr1_tipo['cycle_type'].map({'Base': 85.0, 'Extra': 70.0})

    print("=== KR1 por Tipo de Ciclo ===")
    print(kr1_tipo[['cycle_type', 'vol_original', 'vol_int_any', 'kr1_pct', 'meta']].to_string(index=False))

    vol_total = kr1_tipo['vol_original'].sum()
    vol_alt_total = kr1_tipo['vol_int_any'].sum()
    kr1_total = (1 - vol_alt_total / vol_total) * 100

    print(f"\n=== KR1 Consolidado (OKR ativo): {kr1_total:.1f}% ===")
    print(f"Volume original total: {vol_total:,.0f}")
    print(f"Volume alterado (INT): {vol_alt_total:,.0f}")

    return kr1_tipo, kr1_total


def build_kr1_mes(kr1_coorte):
    kr1_mes = kr1_coorte.groupby(['mes_alvo', 'cycle_type']).agg(
        vol_original=('vol_original', 'sum'),
        vol_int_any=('vol_int_any', 'sum'),
        vol_int_date=('vol_int_date', 'sum'),
        vol_int_cancel=('vol_int_cancel', 'sum'),
        vol_int_grade=('vol_int_grade', 'sum'),
        vol_ext_any=('vol_ext_any', 'sum'),
        vol_ext_cancel=('vol_ext_cancel', 'sum'),
        vol_ext_date_rev=('vol_ext_date_rev', 'sum'),
        n_coortes=('cycle_name', 'nunique'),
    ).reset_index()

    kr1_mes['kr1_pct'] = ((1 - kr1_mes['vol_int_any'] / kr1_mes['vol_original']) * 100).round(1)
    kr1_mes['mes_alvo_str'] = kr1_mes['mes_alvo'].astype(str)

    kr1_mes_total = kr1_coorte.groupby('mes_alvo').agg(
        vol_original=('vol_original', 'sum'),
        vol_int_any=('vol_int_any', 'sum'),
    ).reset_index()
    kr1_mes_total['kr1_pct'] = ((1 - kr1_mes_total['vol_int_any'] / kr1_mes_total['vol_original']) * 100).round(1)
    kr1_mes_total['mes_alvo_str'] = kr1_mes_total['mes_alvo'].astype(str)

    print("=== KR1 por Mês-Alvo ===")
    print(kr1_mes[['mes_alvo_str', 'cycle_type', 'vol_original', 'kr1_pct', 'n_coortes']].to_string(index=False))

    return kr1_mes, kr1_mes_total


def plot_kr1_mes(kr1_mes, colors_cycle):
    fig1 = px.bar(
        kr1_mes,
        x='mes_alvo_str',
        y='kr1_pct',
        color='cycle_type',
        barmode='group',
        color_discrete_map=colors_cycle,
        labels={'kr1_pct': 'KR1 (%)', 'mes_alvo_str': 'Mês-Alvo', 'cycle_type': 'Tipo de Ciclo'},
        title='KR1 — Plan Freeze Rate por Mês-Alvo',
        text='kr1_pct',
    )

    fig1.add_hline(y=85, line_dash="dash", line_color="#1f77b4", opacity=0.5,
                   annotation_text="Meta Base (85%)", annotation_position="top left")
    fig1.add_hline(y=70, line_dash="dash", line_color="#ff7f0e", opacity=0.5,
                   annotation_text="Meta Extra (70%)", annotation_position="bottom left")

    fig1.update_traces(textposition='outside')
    fig1.update_layout(yaxis_range=[0, 105], template='plotly_white')
    fig1.show()


def add_waterfall_reason_code(df_plano):
    df_plano = df_plano.copy()
    df_plano['int_reason_excl'] = np.where(
        df_plano['is_int_cancel'], 'INT_CANCEL',
        np.where(
            df_plano['is_int_date'], 'INT_DATE',
            np.where(
                df_plano['is_int_grade'], 'INT_GRADE',
                'Inalterado'
            )
        )
    )
    return df_plano


def build_waterfall_data(df_plano, kr1_coorte, cycle_type=None):
    df_plano_mes = df_plano.merge(
        kr1_coorte[['cycle_name', 'mes_alvo']].drop_duplicates(),
        on='cycle_name', how='left'
    )

    if cycle_type:
        df_plano_mes = df_plano_mes[df_plano_mes['cycle_type'] == cycle_type]

    wf_excl = df_plano_mes.groupby(['mes_alvo', 'int_reason_excl'])['baseline_planned_qty'].sum().reset_index()
    wf_excl = wf_excl.pivot(index='mes_alvo', columns='int_reason_excl', values='baseline_planned_qty').fillna(0)

    for col in ['INT_CANCEL', 'INT_DATE', 'INT_GRADE', 'Inalterado']:
        if col not in wf_excl.columns:
            wf_excl[col] = 0

    waterfall_data = []
    for mes_alvo in wf_excl.index:
        mes_str = str(mes_alvo)
        vol_orig = wf_excl.loc[mes_alvo].sum()
        v_cancel = wf_excl.loc[mes_alvo, 'INT_CANCEL']
        v_date = wf_excl.loc[mes_alvo, 'INT_DATE']
        v_grade = wf_excl.loc[mes_alvo, 'INT_GRADE']
        v_unchanged = wf_excl.loc[mes_alvo, 'Inalterado']

        waterfall_data.append({'Mês': mes_str, 'Etapa': 'Volume Original', 'Valor': vol_orig, 'Tipo': 'absolute'})
        waterfall_data.append({'Mês': mes_str, 'Etapa': '- INT_CANCEL', 'Valor': -v_cancel, 'Tipo': 'relative'})
        waterfall_data.append({'Mês': mes_str, 'Etapa': '- INT_DATE', 'Valor': -v_date, 'Tipo': 'relative'})
        waterfall_data.append({'Mês': mes_str, 'Etapa': '- INT_GRADE', 'Valor': -v_grade, 'Tipo': 'relative'})
        waterfall_data.append({'Mês': mes_str, 'Etapa': 'Inalterado', 'Valor': v_unchanged, 'Tipo': 'total'})

    return pd.DataFrame(waterfall_data)


def plot_waterfall(df_waterfall, cycle_type=None, mes_escolhido='2026-06'):
    wf_mes = df_waterfall[df_waterfall['Mês'] == mes_escolhido]
    tipo_label = cycle_type if cycle_type else 'Todos'

    fig2 = go.Figure(go.Waterfall(
        name=mes_escolhido,
        orientation="v",
        measure=wf_mes['Tipo'].tolist(),
        x=wf_mes['Etapa'].tolist(),
        y=wf_mes['Valor'].tolist(),
        textposition="outside",
        text=[f"{v:,.0f}" for v in wf_mes['Valor']],
        connector={"line": {"color": "rgb(63, 63, 63)"}},
        increasing={"marker": {"color": "#2ca02c"}},
        decreasing={"marker": {"color": "#d62728"}},
        totals={"marker": {"color": "#1f77b4"}},
    ))

    fig2.update_layout(
        title=f"Waterfall de Alterações Internas — {mes_escolhido} ({tipo_label})<br><sup>Exclusão mútua: CANCEL > DATE > GRADE</sup>",
        yaxis_title="Volume (peças)",
        template='plotly_white',
        showlegend=False,
    )
    fig2.show()


def build_reason_code_data(kr1_mes):
    colors_rc = {
        'INT_DATE': '#d62728',
        'INT_CANCEL': '#e377c2',
        'INT_GRADE': '#ff7f0e',
        'EXT_CANCEL': '#9467bd',
        'EXT_DATE_REV': '#8c564b',
    }

    rc_cols_int = {'vol_int_date': 'INT_DATE', 'vol_int_cancel': 'INT_CANCEL', 'vol_int_grade': 'INT_GRADE'}
    rc_cols_ext = {'vol_ext_cancel': 'EXT_CANCEL', 'vol_ext_date_rev': 'EXT_DATE_REV'}
    all_rc = {**rc_cols_int, **rc_cols_ext}

    kr1_mes_base = kr1_mes[kr1_mes['cycle_type'] == 'Base'].copy()

    rc_data = []
    for _, row in kr1_mes_base.iterrows():
        for col, label in all_rc.items():
            rc_data.append({
                'Mês-Alvo': row['mes_alvo_str'],
                'Tipo Ciclo': row['cycle_type'],
                'Reason Code': label,
                'Volume': row[col],
                'Origem': 'Interno' if label.startswith('INT') else 'Externo',
            })

    df_rc = pd.DataFrame(rc_data)

    vol_mes_base = kr1_mes_base.groupby('mes_alvo_str', as_index=False).agg(
        vol_original=('vol_original', 'sum'),
        vol_int_any=('vol_int_any', 'sum'),
        vol_ext_any=('vol_ext_any', 'sum'),
    )
    vol_mes_map = vol_mes_base.set_index('mes_alvo_str')['vol_original'].to_dict()

    df_rc['pct_vol_original'] = df_rc.apply(
        lambda r: r['Volume'] / vol_mes_map[r['Mês-Alvo']] * 100 if vol_mes_map.get(r['Mês-Alvo'], 0) > 0 else 0,
        axis=1
    )

    pct_total_int = vol_mes_base.copy()
    pct_total_int['pct_alterado'] = (pct_total_int['vol_int_any'] / pct_total_int['vol_original'] * 100).round(1)

    vol_ext_mes = vol_mes_base.copy()
    vol_ext_mes['pct_ext_alterado'] = (vol_ext_mes['vol_ext_any'] / vol_ext_mes['vol_original'] * 100).round(1)

    return df_rc, pct_total_int, vol_ext_mes, colors_rc


def plot_reason_code_breakdown(df_rc, pct_total_int, vol_ext_mes, colors_rc):
    fig3 = px.bar(
        df_rc,
        x='Mês-Alvo',
        y='pct_vol_original',
        color='Reason Code',
        color_discrete_map=colors_rc,
        facet_col='Origem',
        labels={'pct_vol_original': '% do Volume Original', 'Mês-Alvo': 'Mês-Alvo'},
        title='Breakdown de Reason Codes (Ciclos Base) — % do Volume Original por Mês'
              '<br><sup>Barras podem somar > total real (overlap). Linha = % total sem double-count.</sup>',
        barmode='stack',
    )

    fig3.add_trace(
        go.Scatter(
            x=pct_total_int['mes_alvo_str'],
            y=pct_total_int['pct_alterado'],
            mode='lines+markers+text',
            name='% Total Alterado (INT)',
            text=[f"{v:.1f}%" for v in pct_total_int['pct_alterado']],
            textposition='top center',
            line=dict(color='black', width=2, dash='dot', shape='spline'),
            marker=dict(size=7, color='black'),
            showlegend=True,
        ),
        row=1, col=1,
    )

    fig3.add_trace(
        go.Scatter(
            x=vol_ext_mes['mes_alvo_str'],
            y=vol_ext_mes['pct_ext_alterado'],
            mode='lines+markers+text',
            name='% Total Alterado (EXT)',
            text=[f"{v:.1f}%" for v in vol_ext_mes['pct_ext_alterado']],
            textposition='top center',
            line=dict(color='black', width=2, dash='dot', shape='spline'),
            marker=dict(size=7, color='black'),
            showlegend=True,
        ),
        row=1, col=2,
    )

    fig3.update_layout(template='plotly_white')
    fig3.show()


def plot_kr1_coorte_top20(kr1_coorte, colors_cycle):
    top_coortes = kr1_coorte.nlargest(20, 'vol_original').copy()
    top_coortes['mes_alvo_str'] = top_coortes['mes_alvo'].astype(str)

    fig4 = px.bar(
        top_coortes,
        x='cycle_name',
        y='kr1_pct',
        color='cycle_type',
        color_discrete_map=colors_cycle,
        labels={'kr1_pct': 'KR1 (%)', 'cycle_name': 'Ciclo', 'cycle_type': 'Tipo'},
        title='KR1 por Coorte — Top 20 por Volume',
        text='kr1_pct',
        hover_data=['vol_original', 'vol_int_any', 'n_ops', 'mes_alvo_str'],
    )

    fig4.add_hline(y=85, line_dash="dash", line_color="#1f77b4", opacity=0.5)
    fig4.add_hline(y=70, line_dash="dash", line_color="#ff7f0e", opacity=0.5)
    fig4.update_traces(textposition='outside')
    fig4.update_layout(yaxis_range=[0, 105], template='plotly_white', xaxis_tickangle=-45)
    fig4.show()


def prepare_freq_distribution(df_freq):
    df_freq = df_freq.copy()
    df_freq['n_rev_int'] = df_freq['n_rev_planned'] + df_freq['n_rev_grade']
    df_freq['n_rev_ext'] = df_freq['n_rev_reviewed_ext']

    df_freq_op = df_freq.groupby(['op_code', 'cycle_type'], as_index=False).agg(
        n_rev_total=('n_rev_total', 'max'),
    )

    df_freq_op['faixa_rev'] = pd.cut(
        df_freq_op['n_rev_total'],
        bins=[-1, 0, 1, 2, 3, 5, 100],
        labels=['0', '1', '2', '3', '4-5', '6+']
    )

    freq_dist = df_freq_op.groupby(['faixa_rev', 'cycle_type']).agg(
        n_ops=('op_code', 'nunique'),
    ).reset_index()

    pct_sem_rev = (df_freq_op['n_rev_total'] == 0).mean() * 100
    pct_3_mais = (df_freq_op['n_rev_total'] >= 3).mean() * 100

    return df_freq, freq_dist, pct_sem_rev, pct_3_mais


def plot_freq_distribution(freq_dist, colors_cycle):
    fig5 = px.bar(
        freq_dist,
        x='faixa_rev',
        y='n_ops',
        color='cycle_type',
        color_discrete_map=colors_cycle,
        barmode='group',
        labels={'faixa_rev': 'Nº de Revisões', 'n_ops': 'Qtd OPs', 'cycle_type': 'Tipo'},
        title='Distribuição de Frequência de Revisões por OP',
        text='n_ops',
    )

    fig5.update_traces(textposition='outside')
    fig5.update_layout(template='plotly_white')
    fig5.show()


def build_supplier_heatmap(df_freq):
    top_suppliers = df_freq.groupby('supplier_name')['op_code'].count().nlargest(15).index

    df_freq_top = df_freq[df_freq['supplier_name'].isin(top_suppliers)].copy()
    df_freq_top['faixa_rev_ext'] = pd.cut(
        df_freq_top['n_rev_ext'],
        bins=[-1, 0, 1, 2, 100],
        labels=['0', '1', '2', '3+']
    )

    heatmap_data = df_freq_top.groupby(['supplier_name', 'faixa_rev_ext']).size().reset_index(name='count')
    heatmap_pivot = heatmap_data.pivot(index='supplier_name', columns='faixa_rev_ext', values='count').fillna(0)
    heatmap_pct = heatmap_pivot.div(heatmap_pivot.sum(axis=1), axis=0) * 100

    return heatmap_pct


def plot_supplier_heatmap(heatmap_pct):
    fig6 = px.imshow(
        heatmap_pct.values,
        x=heatmap_pct.columns.tolist(),
        y=heatmap_pct.index.tolist(),
        labels=dict(x='Revisões Externas (EXT_DATE_REV)', y='Fornecedor', color='% OP-SKUs'),
        title='Heatmap: Revisões Externas por Fornecedor (% por linha)',
        color_continuous_scale='YlOrRd',
        text_auto='.1f',
        aspect='auto',
        zmin=0,
        zmax=100,
    )
    fig6.update_layout(template='plotly_white')
    fig6.show()


def prepare_instability_data(df_freq):
    df_freq = df_freq.copy()
    df_freq['dominant_type'] = np.where(
        df_freq['n_rev_int'] >= df_freq['n_rev_ext'],
        'Interno (INT)',
        'Externo (EXT)'
    )
    df_freq['total_magnitude'] = df_freq['total_magnitude_dt_planned'] + df_freq['total_magnitude_dt_reviewed']

    df_freq_rev = df_freq[df_freq['n_rev_total'] > 0].copy()
    df_freq_ext = df_freq[df_freq['n_rev_reviewed_ext'] > 0].copy()

    df_freq_ext_op = df_freq_ext.groupby(
        ['supplier_name', 'product_name', 'cycle_type', 'op_code'],
        as_index=False
    ).agg(
        n_rev_ext_op=('n_rev_reviewed_ext', 'max'),
        total_dias_deslocamento_ext_op=('total_magnitude_dt_reviewed', 'max'),
    )

    df_supplier_product_ext = df_freq_ext_op.groupby(
        ['supplier_name', 'product_name', 'cycle_type'],
        as_index=False
    ).agg(
        n_ops_mudadas=('op_code', 'nunique'),
        total_dias_deslocamento_ext=('total_dias_deslocamento_ext_op', 'sum'),
        total_rev_ext=('n_rev_ext_op', 'sum'),
    )

    df_supplier_product_ext['dias_deslocamento_medio'] = (
        df_supplier_product_ext['total_dias_deslocamento_ext'] / df_supplier_product_ext['total_rev_ext']
    )
    df_supplier_product_ext['n_rev_ext_medio'] = (
        df_supplier_product_ext['total_rev_ext'] / df_supplier_product_ext['n_ops_mudadas']
    )

    df_supplier_product_ext['dias_deslocamento_medio'] = df_supplier_product_ext['dias_deslocamento_medio'].round(1)
    df_supplier_product_ext['n_rev_ext_medio'] = df_supplier_product_ext['n_rev_ext_medio'].round(2)

    return df_freq, df_freq_rev, df_freq_ext, df_supplier_product_ext


def plot_instability_scatter(df_supplier_product_ext, colors_cycle):
    fig7 = px.scatter(
        df_supplier_product_ext,
        x='n_ops_mudadas',
        y='dias_deslocamento_medio',
        size='n_rev_ext_medio',
        color='cycle_type',
        color_discrete_map=colors_cycle,
        labels={
            'n_ops_mudadas': 'Nº de OPs Mudadas',
            'dias_deslocamento_medio': 'Dias Médios de Deslocamento',
            'n_rev_ext_medio': 'Nº Médio de Revisões Externas por OP',
            'cycle_type': 'Tipo de Ciclo',
        },
        title='Scatter de Alterações Externas — Supplier × Product_Name',
        hover_data=['supplier_name', 'product_name', 'total_rev_ext'],
        size_max=30,
        opacity=0.65,
    )

    fig7.update_layout(template='plotly_white')
    fig7.show()


def plot_instability_scatter_by_cycle(df_supplier_product_ext, cycle_type):
    df_cycle = df_supplier_product_ext[df_supplier_product_ext['cycle_type'] == cycle_type].copy()

    fig = px.scatter(
        df_cycle,
        x='n_ops_mudadas',
        y='dias_deslocamento_medio',
        size='n_rev_ext_medio',
        color='supplier_name',
        labels={
            'n_ops_mudadas': 'Nº de OPs Mudadas',
            'dias_deslocamento_medio': 'Dias Médios de Deslocamento',
            'n_rev_ext_medio': 'Nº Médio de Revisões Externas por OP',
            'supplier_name': 'Fornecedor',
        },
        title=f'Scatter de Alterações Externas — Supplier × Product_Name ({cycle_type})',
        hover_data=['supplier_name', 'product_name', 'total_rev_ext'],
        size_max=30,
        opacity=0.7,
    )

    fig.update_layout(template='plotly_white')
    fig.show()


def build_external_treemap_data(df_supplier_product_ext):
    df_treemap = df_supplier_product_ext.copy()
    df_treemap = df_treemap.groupby(
        ['supplier_name', 'product_name'],
        as_index=False
    ).agg(
        n_ops_mudadas=('n_ops_mudadas', 'sum'),
        total_rev_ext=('total_rev_ext', 'sum'),
        total_dias_deslocamento_ext=('total_dias_deslocamento_ext', 'sum'),
    )

    df_treemap['dias_deslocamento_medio'] = (
        df_treemap['total_dias_deslocamento_ext'] / df_treemap['total_rev_ext']
    ).round(1)
    df_treemap['n_rev_ext_medio'] = (
        df_treemap['total_rev_ext'] / df_treemap['n_ops_mudadas']
    ).round(2)

    return df_treemap


def plot_external_treemap(df_treemap):
    fig = px.treemap(
        df_treemap,
        path=['supplier_name', 'product_name'],
        values='n_ops_mudadas',
        color='supplier_name',
        custom_data=['dias_deslocamento_medio', 'n_rev_ext_medio', 'total_rev_ext'],
        title='Treemap de Alterações Externas: Fornecedor → Produto',
    )

    fig.update_traces(
        texttemplate='<b>%{label}</b><br>%{value} OPs<br>%{customdata[0]:.1f} dias<br>%{customdata[1]:.2f} rev/OP',
        hovertemplate='<b>%{label}</b><br>'
                      'OPs alteradas: %{value}<br>'
                      'Dias médios de deslocamento: %{customdata[0]:.1f}<br>'
                      'Nº médio de revisões por OP: %{customdata[1]:.2f}<br>'
                      'Total de revisões externas: %{customdata[2]}<extra></extra>'
    )

    fig.update_layout(template='plotly_white')
    fig.show()


def load_timeline_drilldown(df_freq_rev, sql_path):
    top_instavel = df_freq_rev.nlargest(1, 'n_rev_total').iloc[0]
    op_drill = top_instavel['op_code']
    sku_drill = top_instavel['product_sku']

    print(f"Drill-down: OP {op_drill}, SKU {sku_drill}")
    print(f"  Fornecedor: {top_instavel['supplier_name']}")
    print(f"  Ciclo: {top_instavel['cycle_name']}")
    print(f"  Revisões: {int(top_instavel['n_rev_total'])} (INT_DATE: {int(top_instavel['n_rev_planned'])}, EXT_DATE_REV: {int(top_instavel['n_rev_reviewed_ext'])}, INT_GRADE: {int(top_instavel['n_rev_grade'])})")

    df_timeline = convert_sql_to_df(
        sql_path + 'timeline_revisoes.sql',
        op_code=op_drill,
        product_sku=sku_drill,
    )

    return top_instavel, op_drill, sku_drill, df_timeline


def prepare_timeline_data(df_timeline):
    df_timeline = df_timeline.copy()
    df_timeline['ingestion_date'] = pd.to_datetime(df_timeline['ingestion_date'])
    df_timeline['dt_planned_entry_warehouse'] = pd.to_datetime(df_timeline['dt_planned_entry_warehouse'])
    df_timeline['dt_reviewed_entry_warehouse'] = pd.to_datetime(df_timeline['dt_reviewed_entry_warehouse'])
    return df_timeline


def plot_timeline_datas(df_timeline, op_drill, sku_drill):
    fig8 = go.Figure()

    fig8.add_trace(go.Scatter(
        x=df_timeline['ingestion_date'],
        y=df_timeline['dt_planned_entry_warehouse'],
        mode='lines+markers',
        name='dt_planned',
        line=dict(color='#d62728', width=2),
        marker=dict(
            size=8,
            color=['#d62728' if c == 'INT_DATE' else 'rgba(0,0,0,0)' for c in df_timeline['change_dt_planned'].fillna('')],
            line=dict(width=1, color='black'),
        ),
    ))

    fig8.add_trace(go.Scatter(
        x=df_timeline['ingestion_date'],
        y=df_timeline['dt_reviewed_entry_warehouse'],
        mode='lines+markers',
        name='dt_reviewed',
        line=dict(color='#9467bd', width=2),
        marker=dict(
            size=8,
            color=['#9467bd' if c == 'EXT_DATE_REV' else 'rgba(0,0,0,0)' for c in df_timeline['change_dt_reviewed'].fillna('')],
            line=dict(width=1, color='black'),
        ),
    ))

    has_grade_change = df_timeline[df_timeline['change_grade'] == 'INT_GRADE']
    if len(has_grade_change) > 0:
        fig8.add_trace(go.Scatter(
            x=has_grade_change['ingestion_date'],
            y=has_grade_change['dt_planned_entry_warehouse'],
            mode='markers',
            name='INT_GRADE',
            marker=dict(size=14, color='#ff7f0e', symbol='diamond', line=dict(width=2, color='black')),
        ))

    fig8.update_layout(
        title=f'Timeline de Revisões — OP {op_drill} / SKU {sku_drill}',
        xaxis_title='Data do Snapshot',
        yaxis_title='Data Planejada/Revisada',
        template='plotly_white',
        hovermode='x unified',
    )
    fig8.show()


def plot_timeline_grade(df_timeline, op_drill, sku_drill):
    fig9 = go.Figure()

    fig9.add_trace(go.Scatter(
        x=df_timeline['ingestion_date'],
        y=df_timeline['planned_quantity'],
        mode='lines+markers',
        name='planned_quantity',
        line=dict(color='#ff7f0e', width=2),
        marker=dict(
            size=8,
            color=['#ff7f0e' if c == 'INT_GRADE' else 'rgba(0,0,0,0.1)' for c in df_timeline['change_grade'].fillna('')],
            line=dict(width=1, color='black'),
        ),
        fill='tozeroy',
        fillcolor='rgba(255,127,14,0.1)',
    ))

    fig9.update_layout(
        title=f'Evolução da Grade (planned_quantity) — OP {op_drill} / SKU {sku_drill}',
        xaxis_title='Data do Snapshot',
        yaxis_title='Quantidade Planejada',
        template='plotly_white',
    )
    fig9.show()


def load_evolucao(sql_path, ciclos_excluidos, ciclos_validos, kr1_coorte):
    df_evolucao = convert_sql_to_df(sql_path + 'kr1_evolucao.sql')
    df_evolucao = df_evolucao[~df_evolucao['cycle_name'].isin(ciclos_excluidos)].copy()
    print(f"Evolução temporal: {df_evolucao.shape[0]:,} linhas (semana × ciclo × fornecedor × produto)")
    print(f"Semanas: {df_evolucao['snapshot_week'].nunique()}, Ciclos: {df_evolucao['cycle_name'].nunique()}, Fornecedores: {df_evolucao['supplier_name'].nunique()}, Produtos: {df_evolucao['product_name'].nunique()}")

    df_evolucao = df_evolucao[df_evolucao['cycle_name'].isin(ciclos_validos)]
    df_evolucao = df_evolucao.merge(
        kr1_coorte[['cycle_name', 'mes_alvo']].drop_duplicates(),
        on='cycle_name', how='left'
    )

    df_evolucao['kr1'] = 1 - (df_evolucao['vol_int_any'] / df_evolucao['vol_original'])
    df_evolucao['kr1_pct'] = (df_evolucao['kr1'] * 100).round(1)
    df_evolucao['snapshot_week'] = pd.to_datetime(df_evolucao['snapshot_week'])
    df_evolucao['mes_alvo_str'] = df_evolucao['mes_alvo'].astype(str)

    print(f"\nApós filtro: {df_evolucao.shape[0]:,} linhas, {df_evolucao['cycle_name'].nunique()} ciclos")
    return df_evolucao


def build_evol_mes(df_evolucao):
    evol_mes = df_evolucao.groupby(['snapshot_week', 'mes_alvo_str', 'cycle_type']).agg(
        vol_original=('vol_original', 'sum'),
        vol_int_any=('vol_int_any', 'sum'),
    ).reset_index()
    evol_mes['kr1_pct'] = ((1 - evol_mes['vol_int_any'] / evol_mes['vol_original']) * 100).round(1)
    return evol_mes


def plot_evol_kr1_mes(evol_mes):
    fig_a = px.line(
        evol_mes.sort_values('snapshot_week'),
        x='snapshot_week', y='kr1_pct', color='mes_alvo_str',
        facet_col='cycle_type',
        title='Evolução do KR1 por Mês-Alvo — Curva de Degradação (Base vs Extra)',
        labels={'snapshot_week': 'Semana', 'kr1_pct': 'KR1 (%)', 'mes_alvo_str': 'Mês-Alvo'},
        markers=True,
        line_shape='spline'
    )
    fig_a.add_hline(y=85, line_dash="dot", line_color="green", annotation_text="Meta 85%", col=1)
    fig_a.add_hline(y=70, line_dash="dot", line_color="orange", annotation_text="Meta 70%", col=2)
    fig_a.update_layout(yaxis_range=[0, 105], template='plotly_white')
    fig_a.update_yaxes(range=[0, 105])
    fig_a.show()


def build_evol_tipo(df_evolucao):
    evol_tipo = df_evolucao.groupby(['snapshot_week', 'cycle_type']).agg(
        vol_original=('vol_original', 'sum'),
        vol_int_any=('vol_int_any', 'sum'),
    ).reset_index()
    evol_tipo['kr1_pct'] = ((1 - evol_tipo['vol_int_any'] / evol_tipo['vol_original']) * 100).round(1)
    return evol_tipo


def plot_evol_kr1_tipo(evol_tipo):
    fig_b = px.line(
        evol_tipo.sort_values('snapshot_week'),
        x='snapshot_week', y='kr1_pct', color='cycle_type',
        title='Evolução do KR1 Consolidado — Base vs Extra',
        labels={'snapshot_week': 'Semana', 'kr1_pct': 'KR1 (%)', 'cycle_type': 'Tipo de Ciclo'},
        markers=True,
        line_shape='spline',
        color_discrete_map={'Base': '#636EFA', 'Extra': '#EF553B'}
    )
    fig_b.add_hline(y=85, line_dash="dot", line_color="#636EFA", annotation_text="Meta Base (85%)")
    fig_b.add_hline(y=70, line_dash="dot", line_color="#EF553B", annotation_text="Meta Extra (70%)")
    fig_b.update_layout(yaxis_range=[0, 105], template='plotly_white')
    fig_b.show()


def build_evol_heatmap_base(evol_mes):
    evol_mes_base = evol_mes[evol_mes['cycle_type'] == 'Base'].copy()
    heatmap_pivot = evol_mes_base.pivot(
        index='mes_alvo_str',
        columns='snapshot_week',
        values='kr1_pct'
    )
    heatmap_pivot = heatmap_pivot.sort_index()
    heatmap_pivot.columns = [c.strftime('%d/%m') for c in heatmap_pivot.columns]
    return heatmap_pivot


def plot_evol_heatmap_base(heatmap_pivot):
    fig_c = px.imshow(
        heatmap_pivot,
        title='KR1 (%) — Semana × Mês-Alvo [Ciclos Base]',
        labels=dict(x='Semana do Snapshot', y='Mês-Alvo', color='KR1 (%)'),
        color_continuous_scale='RdYlGn',
        zmin=50, zmax=100,
        text_auto='.0f',
        aspect='auto'
    )
    fig_c.update_layout(template='plotly_white')
    fig_c.show()


def build_evol_ext_mes(df_evolucao):
    evol_ext_mes = df_evolucao.groupby(['snapshot_week', 'mes_alvo_str', 'cycle_type']).agg(
        vol_original=('vol_original', 'sum'),
        vol_ext_any=('vol_ext_any', 'sum'),
        vol_ext_cancel=('vol_ext_cancel', 'sum'),
        vol_ext_date_rev=('vol_ext_date_rev', 'sum'),
    ).reset_index()
    evol_ext_mes['pct_ext_any'] = (evol_ext_mes['vol_ext_any'] / evol_ext_mes['vol_original'] * 100).round(1)
    return evol_ext_mes


def plot_evol_ext_mes(evol_ext_mes):
    fig_d = px.line(
        evol_ext_mes.sort_values('snapshot_week'),
        x='snapshot_week', y='pct_ext_any', color='mes_alvo_str',
        facet_col='cycle_type',
        title='Evolução das Alterações Externas por Mês-Alvo (Base vs Extra)',
        labels={'snapshot_week': 'Semana', 'pct_ext_any': '% Volume Impactado (EXT)', 'mes_alvo_str': 'Mês-Alvo'},
        markers=True,
        line_shape='spline'
    )
    fig_d.update_layout(template='plotly_white')
    fig_d.update_yaxes(rangemode='tozero')
    fig_d.show()


def build_evol_ext_tipo(df_evolucao):
    evol_ext_tipo = df_evolucao.groupby(['snapshot_week', 'cycle_type']).agg(
        vol_original=('vol_original', 'sum'),
        vol_ext_any=('vol_ext_any', 'sum'),
        vol_ext_cancel=('vol_ext_cancel', 'sum'),
        vol_ext_date_rev=('vol_ext_date_rev', 'sum'),
    ).reset_index()
    evol_ext_tipo['pct_ext_any'] = (evol_ext_tipo['vol_ext_any'] / evol_ext_tipo['vol_original'] * 100).round(1)
    return evol_ext_tipo


def plot_evol_ext_tipo(evol_ext_tipo):
    fig_e = px.line(
        evol_ext_tipo.sort_values('snapshot_week'),
        x='snapshot_week', y='pct_ext_any', color='cycle_type',
        title='Evolução Consolidada das Alterações Externas — Base vs Extra',
        labels={'snapshot_week': 'Semana', 'pct_ext_any': '% Volume Impactado (EXT)', 'cycle_type': 'Tipo de Ciclo'},
        markers=True,
        line_shape='spline',
        color_discrete_map={'Base': '#636EFA', 'Extra': '#EF553B'}
    )
    fig_e.update_layout(template='plotly_white')
    fig_e.update_yaxes(rangemode='tozero')
    fig_e.show()


def build_evol_ext_heatmap_base(evol_ext_mes):
    evol_ext_base = evol_ext_mes[evol_ext_mes['cycle_type'] == 'Base'].copy()
    heatmap_ext_pivot = evol_ext_base.pivot(
        index='mes_alvo_str',
        columns='snapshot_week',
        values='pct_ext_any'
    )
    heatmap_ext_pivot = heatmap_ext_pivot.sort_index()
    heatmap_ext_pivot.columns = [c.strftime('%d/%m') for c in heatmap_ext_pivot.columns]
    return heatmap_ext_pivot


def plot_evol_ext_heatmap_base(heatmap_ext_pivot):
    fig_f = px.imshow(
        heatmap_ext_pivot,
        title='Alterações Externas (%) — Semana × Mês-Alvo [Ciclos Base]',
        labels=dict(x='Semana do Snapshot', y='Mês-Alvo', color='% Impactado (EXT)'),
        color_continuous_scale='YlOrRd',
        zmin=0,
        text_auto='.1f',
        aspect='auto'
    )
    fig_f.update_layout(template='plotly_white')
    fig_f.show()


def export_outputs(output_path, today, kr1_coorte, kr1_mes, df_plano, df_freq, df_evolucao):
    kr1_coorte.to_csv(f'{output_path}kr1_por_coorte_{today}.csv', index=False)
    kr1_mes.to_csv(f'{output_path}kr1_por_mes_{today}.csv', index=False)
    df_plano.to_csv(f'{output_path}plano_vs_atual_{today}.csv', index=False)
    df_freq.to_csv(f'{output_path}frequencia_revisoes_{today}.csv', index=False)
    df_evolucao.to_csv(f'{output_path}kr1_evolucao_{today}.csv', index=False)

    print(f"Arquivos exportados em {output_path} com sufixo _{today}")

# =============================================================================
# OKR PONDERADO E VISÕES INTERNAS ADICIONAIS
# =============================================================================

OKR_WEIGHTS = {0: 35, 1: 35, 2: 20, 3: 10}


def _period_add(period, n):
    return period + n


def build_weighted_okr_evolution(
    df_evolucao,
    group_cols=None,
    include_cycle_type=True,
    weights=None,
    min_weight_coverage=100,
):
    """Calcula OKR ponderado Cn..Cn+3 ao longo do tempo.

    Em cada snapshot_week, Cn é o mês do snapshot. O OKR é:
    (35*taxa Cn + 35*taxa Cn+1 + 20*taxa Cn+2 + 10*taxa Cn+3)/100.
    A taxa de cada ciclo/mês é calculada apenas com alterações internas.
    """
    if weights is None:
        weights = OKR_WEIGHTS
    if group_cols is None:
        group_cols = []

    d = df_evolucao.copy()
    d['snapshot_week'] = pd.to_datetime(d['snapshot_week'])
    d['snapshot_month'] = d['snapshot_week'].dt.to_period('M')
    d['mes_period'] = pd.PeriodIndex(d['mes_alvo_str'], freq='M')

    dims = ['snapshot_week', 'snapshot_month']
    if include_cycle_type:
        dims.append('cycle_type')
    dims.extend(group_cols)
    dims.append('mes_period')

    monthly = d.groupby(dims, dropna=False).agg(
        vol_original=('vol_original', 'sum'),
        vol_int_any=('vol_int_any', 'sum'),
    ).reset_index()
    monthly['kr1_pct'] = np.where(
        monthly['vol_original'] > 0,
        (1 - monthly['vol_int_any'] / monthly['vol_original']) * 100,
        np.nan,
    )

    context_cols = ['snapshot_week', 'snapshot_month']
    if include_cycle_type:
        context_cols.append('cycle_type')
    context_cols.extend(group_cols)

    records = []
    for ctx, grp in monthly.groupby(context_cols, dropna=False):
        if not isinstance(ctx, tuple):
            ctx = (ctx,)
        ctx_record = dict(zip(context_cols, ctx))
        snapshot_month = ctx_record['snapshot_month']
        by_month = grp.set_index('mes_period')

        weighted_sum = 0.0
        coverage = 0
        volume_window = 0.0
        component_rates = {}
        component_volumes = {}

        for offset, weight in weights.items():
            target_month = _period_add(snapshot_month, offset)
            label = f'c{offset}'
            if target_month in by_month.index:
                row = by_month.loc[target_month]
                if isinstance(row, pd.DataFrame):
                    row = row.iloc[0]
                rate = row['kr1_pct']
                volume = row['vol_original']
                component_rates[f'{label}_mes_alvo'] = str(target_month)
                component_rates[f'{label}_kr1_pct'] = rate
                component_volumes[f'{label}_vol_original'] = volume
                if pd.notna(rate):
                    weighted_sum += rate * weight
                    coverage += weight
                    volume_window += volume
            else:
                component_rates[f'{label}_mes_alvo'] = str(target_month)
                component_rates[f'{label}_kr1_pct'] = np.nan
                component_volumes[f'{label}_vol_original'] = 0

        okr_weighted = weighted_sum / 100 if coverage >= min_weight_coverage else np.nan
        records.append({
            **ctx_record,
            **component_rates,
            **component_volumes,
            'weight_coverage': coverage,
            'vol_original_window': volume_window,
            'okr_weighted_pct': okr_weighted,
        })

    return pd.DataFrame(records).sort_values(context_cols)


def plot_weighted_okr_evolution(okr_evol_tipo):
    fig = px.line(
        okr_evol_tipo.dropna(subset=['okr_weighted_pct']),
        x='snapshot_week',
        y='okr_weighted_pct',
        color='cycle_type',
        markers=True,
        line_shape='spline',
        labels={
            'snapshot_week': 'Semana do Snapshot',
            'okr_weighted_pct': 'OKR Ponderado (%)',
            'cycle_type': 'Tipo de Ciclo',
        },
        title='Evolução do OKR Ponderado — Cn..Cn+3 (Base vs Extra)'
              '<br><sup>OKR = (35*C0 + 35*C1 + 20*C2 + 10*C3)/100; apenas alterações internas</sup>',
        color_discrete_map=colors_cycle,
    )
    fig.update_layout(template='plotly_white', yaxis_range=[0, 105])
    fig.show()




def build_allocated_volume_stack(df_plano, kr1_coorte, cycle_type=None):
    d = df_plano.merge(
        kr1_coorte[['cycle_name', 'mes_alvo']].drop_duplicates(),
        on='cycle_name', how='left'
    ).copy()
    if cycle_type:
        d = d[d['cycle_type'] == cycle_type]

    if 'int_reason_excl' not in d.columns:
        d = add_waterfall_reason_code(d)

    labels = {
        'INT_CANCEL': 'Cancelamento',
        'INT_DATE': 'Alteração de Data',
        'INT_GRADE': 'Alteração de Grade',
        'Inalterado': 'Inalterado',
    }
    reason_order = ['Cancelamento', 'Alteração de Data', 'Alteração de Grade', 'Inalterado']
    cycle_order = ['Base', 'Extra']

    d['reason_label'] = d['int_reason_excl'].map(labels).fillna(d['int_reason_excl'])
    d['reason_label'] = pd.Categorical(d['reason_label'], categories=reason_order, ordered=True)
    d['cycle_type'] = pd.Categorical(d['cycle_type'], categories=cycle_order, ordered=True)
    d['mes_alvo_str'] = d['mes_alvo'].astype(str)

    out = d.groupby(['mes_alvo_str', 'cycle_type', 'reason_label'], observed=True, as_index=False).agg(
        volume=('baseline_planned_qty', 'sum'),
        qtd_ops=('op_code', 'nunique'),
    )
    out = out[out['volume'] > 0].copy()
    out['mes_alvo_str'] = pd.Categorical(
        out['mes_alvo_str'],
        categories=sorted(out['mes_alvo_str'].dropna().unique()),
        ordered=True,
    )
    return out.sort_values(['mes_alvo_str', 'cycle_type', 'reason_label'])




def plot_allocated_volume_stack(stack_data):
    colors = {
        'Cancelamento': '#e377c2',
        'Alteração de Data': '#d62728',
        'Alteração de Grade': '#ff7f0e',
        'Inalterado': '#1f77b4',
    }
    reason_order = ['Cancelamento', 'Alteração de Data', 'Alteração de Grade', 'Inalterado']
    cycle_order = ['Base', 'Extra']
    opacity_by_cycle = {'Base': 1.0, 'Extra': 0.38}

    plot_df = stack_data.copy()
    plot_df['mes_alvo_str'] = plot_df['mes_alvo_str'].astype(str)
    months = sorted(plot_df['mes_alvo_str'].dropna().unique())

    fig = go.Figure()
    for cycle_type in cycle_order:
        for reason in reason_order:
            trace_df = (
                plot_df[
                    (plot_df['cycle_type'].astype(str) == cycle_type)
                    & (plot_df['reason_label'].astype(str) == reason)
                ]
                .set_index('mes_alvo_str')
                .reindex(months)
                .reset_index()
            )
            y = trace_df['volume'].fillna(0)
            text = [f'{v:,.0f}' if v > 0 else '' for v in y]

            # Eixo X multicategoria: para cada mês, há uma barra Base e outra Extra lado a lado.
            x_multi = [trace_df['mes_alvo_str'].tolist(), [cycle_type] * len(trace_df)]

            fig.add_trace(
                go.Bar(
                    x=x_multi,
                    y=y,
                    name=reason,
                    legendgroup=reason,
                    showlegend=(cycle_type == 'Base'),
                    marker=dict(
                        color=colors[reason],
                        opacity=opacity_by_cycle[cycle_type],
                    ),
                    text=text,
                    textposition='inside',
                    insidetextanchor='middle',
                    textangle=0,
                    customdata=np.column_stack([
                        np.repeat(cycle_type, len(trace_df)),
                        np.repeat(reason, len(trace_df)),
                        trace_df['qtd_ops'].fillna(0),
                    ]),
                    hovertemplate=(
                        'Mês: %{x[0]}<br>'
                        'Tipo de Ciclo: %{customdata[0]}<br>'
                        'Classificação: %{customdata[1]}<br>'
                        'Volume: %{y:,.0f} peças<br>'
                        'Qtd OPs: %{customdata[2]:,.0f}'
                        '<extra></extra>'
                    ),
                )
            )

    fig.update_layout(
        title='Volume Alocado por Ciclo de Alocação — Alterações Internas Exclusivas'
              '<br><sup>Base e Extra lado a lado por mês; Extra com opacidade reduzida. Exclusão mútua: CANCEL > DATE > GRADE > Inalterado</sup>',
        xaxis_title='Mês correspondente ao ciclo de alocação',
        yaxis_title='Volume alocado (peças)',
        barmode='relative',
        bargap=0.18,
        bargroupgap=0.06,
        template='plotly_white',
        legend_title='Classificação',
    )
    fig.update_xaxes(categoryorder='array', categoryarray=months, tickangle=-45)

    max_total = (
        plot_df.groupby(['mes_alvo_str', 'cycle_type'], observed=True)['volume'].sum().max()
        if not plot_df.empty else 0
    )
    if max_total and pd.notna(max_total):
        fig.update_yaxes(range=[0, max_total * 1.18])

    fig.add_annotation(
        x=1.01,
        y=0.96,
        xref='paper',
        yref='paper',
        text='Base: opacidade 100%<br>Extra: opacidade 38%',
        showarrow=False,
        align='left',
        font=dict(size=12),
    )

    fig.show()


def build_internal_change_pct_by_month(kr1_mes):
    records = []
    rc_cols = {
        'vol_int_cancel': 'Cancelamento',
        'vol_int_date': 'Alteração de Data',
        'vol_int_grade': 'Alteração de Grade',
    }
    for _, row in kr1_mes.iterrows():
        for col, label in rc_cols.items():
            records.append({
                'mes_alvo_str': row['mes_alvo_str'],
                'cycle_type': row['cycle_type'],
                'reason_label': label,
                'pct_vol_original': row[col] / row['vol_original'] * 100 if row['vol_original'] else 0,
            })
    bars = pd.DataFrame(records)

    line = kr1_mes.copy()
    line['pct_int_any'] = (line['vol_int_any'] / line['vol_original'] * 100).round(1)
    return bars, line


def plot_internal_change_pct_by_month(bars, line):
    colors = {
        'Cancelamento': '#e377c2',
        'Alteração de Data': '#d62728',
        'Alteração de Grade': '#ff7f0e',
    }
    fig = px.bar(
        bars,
        x='mes_alvo_str',
        y='pct_vol_original',
        color='reason_label',
        facet_col='cycle_type',
        category_orders={'reason_label': list(colors.keys())},
        color_discrete_map=colors,
        labels={
            'mes_alvo_str': 'Mês-Alvo',
            'pct_vol_original': '% do Volume Original',
            'reason_label': 'Alteração Interna',
            'cycle_type': 'Tipo de Ciclo',
        },
        title='Percentual de Alteração Interna por Mês — Data, Cancelamento e Grade'
              '<br><sup>Linha = % total alterado internamente sem double-count; sem alterações externas</sup>',
        barmode='stack',
    )

    cycle_types = list(line['cycle_type'].dropna().unique())
    for idx, cycle_type in enumerate(cycle_types, start=1):
        l = line[line['cycle_type'] == cycle_type]
        fig.add_trace(
            go.Scatter(
                x=l['mes_alvo_str'],
                y=l['pct_int_any'],
                mode='lines+markers+text',
                name=f'% Total INT — {cycle_type}',
                text=[f'{v:.1f}%' for v in l['pct_int_any']],
                textposition='top center',
                line=dict(color='black', width=2, dash='dot', shape='spline'),
                marker=dict(size=7, color='black'),
                showlegend=idx == 1,
            ),
            row=1,
            col=idx,
        )

    fig.update_layout(template='plotly_white')
    fig.update_xaxes(tickangle=-45)
    fig.show()



def plot_weighted_okr_by_dimension(okr_evol_dim, dimension_col, title, top_n=None):
    plot_df = okr_evol_dim.dropna(subset=['okr_weighted_pct']).copy()
    has_cycle_type = 'cycle_type' in plot_df.columns
    subtitle = 'Todas as séries com cobertura completa; apenas alterações internas'

    if top_n is not None:
        latest_week = okr_evol_dim['snapshot_week'].max()
        top_values = (
            okr_evol_dim[okr_evol_dim['snapshot_week'] == latest_week]
            .dropna(subset=['okr_weighted_pct'])
            .groupby(dimension_col)['vol_original_window'].sum()
            .nlargest(top_n)
            .index
        )
        plot_df = plot_df[plot_df[dimension_col].isin(top_values)]
        if has_cycle_type:
            subtitle = f'Top {top_n} por volume na janela OKR atual; Base e Extra separados; apenas alterações internas'
        else:
            subtitle = f'Top {top_n} por volume na janela OKR atual; apenas ciclos Base; apenas alterações internas'

    line_args = {
        'data_frame': plot_df,
        'x': 'snapshot_week',
        'y': 'okr_weighted_pct',
        'color': dimension_col,
        'markers': True,
        'line_shape': 'spline',
        'labels': {
            'snapshot_week': 'Semana do Snapshot',
            'okr_weighted_pct': 'OKR Ponderado (%)',
            dimension_col: dimension_col,
            'cycle_type': 'Tipo de Ciclo',
        },
        'title': title + f'<br><sup>{subtitle}</sup>',
    }
    if has_cycle_type:
        line_args['line_dash'] = 'cycle_type'
        line_args['category_orders'] = {'cycle_type': ['Base', 'Extra']}

    fig = px.line(**line_args)
    fig.update_layout(template='plotly_white', yaxis_range=[0, 105])
    fig.show()



def build_current_supplier_volume_okr(df_evolucao, top_n=20):
    okr_supplier = build_weighted_okr_evolution(
        df_evolucao,
        group_cols=['supplier_name'],
        include_cycle_type=True,
        min_weight_coverage=100,
    ).dropna(subset=['okr_weighted_pct']).copy()

    latest_by_cycle = okr_supplier.groupby('cycle_type')['snapshot_week'].transform('max')
    current = okr_supplier[okr_supplier['snapshot_week'] == latest_by_cycle].copy()

    top_suppliers = (
        current.groupby('supplier_name')['vol_original_window']
        .sum()
        .nlargest(top_n)
        .index
    )
    current = current[current['supplier_name'].isin(top_suppliers)].copy()
    current['supplier_name'] = pd.Categorical(current['supplier_name'], categories=top_suppliers, ordered=True)
    current['cycle_type'] = pd.Categorical(current['cycle_type'], categories=['Base', 'Extra'], ordered=True)
    return current.sort_values(['supplier_name', 'cycle_type'])

def plot_current_supplier_volume_okr(current_supplier_okr):
    plot_df = current_supplier_okr.copy().sort_values(['supplier_name', 'cycle_type'])

    fig = make_subplots(specs=[[{'secondary_y': True}]])
    for cycle_type in ['Base', 'Extra']:
        d = plot_df[plot_df['cycle_type'].astype(str) == cycle_type]
        opacity = 1.0 if cycle_type == 'Base' else 0.45
        fig.add_bar(
            x=d['supplier_name'].astype(str),
            y=d['vol_original_window'],
            name=f'Volume Alocado — {cycle_type}',
            marker_color=colors_cycle.get(cycle_type, '#9bd3f5'),
            opacity=opacity,
            text=[f'{v:,.0f}' for v in d['vol_original_window']],
            textposition='outside',
            secondary_y=False,
        )
        fig.add_trace(
            go.Scatter(
                x=d['supplier_name'].astype(str),
                y=d['okr_weighted_pct'],
                name=f'OKR Atual — {cycle_type} (%)',
                mode='lines+markers+text',
                text=[f'{v:.0f}%' for v in d['okr_weighted_pct']],
                textposition='top center',
                line=dict(
                    color=colors_cycle.get(cycle_type, 'black'),
                    width=2,
                    dash='solid' if cycle_type == 'Base' else 'dash',
                    shape='spline',
                ),
                marker=dict(color=colors_cycle.get(cycle_type, 'black'), size=7),
            ),
            secondary_y=True,
        )

    fig.update_layout(
        title='Volume Alocado e OKR Atual por Fornecedor — Base vs Extra'
              '<br><sup>Ordenado por volume alocado na janela Cn..Cn+3; apenas alterações internas</sup>',
        template='plotly_white',
        xaxis_tickangle=-45,
        barmode='group',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    )
    fig.update_yaxes(title_text='Volume Alocado (peças)', secondary_y=False)
    fig.update_yaxes(title_text='OKR Atual (%)', range=[0, 105], secondary_y=True)
    fig.show()



def build_current_product_volume_kr(df_evolucao, top_n=10):
    okr_product = build_weighted_okr_evolution(
        df_evolucao,
        group_cols=['product_name'],
        include_cycle_type=True,
        min_weight_coverage=100,
    ).dropna(subset=['okr_weighted_pct']).copy()

    latest_by_cycle = okr_product.groupby('cycle_type')['snapshot_week'].transform('max')
    current = okr_product[okr_product['snapshot_week'] == latest_by_cycle].copy()

    top_products = (
        current.groupby('product_name')['vol_original_window']
        .sum()
        .nlargest(top_n)
        .index
    )
    current = current[current['product_name'].isin(top_products)].copy()
    current['product_name'] = pd.Categorical(current['product_name'], categories=top_products, ordered=True)
    current['cycle_type'] = pd.Categorical(current['cycle_type'], categories=['Base', 'Extra'], ordered=True)
    return current.sort_values(['product_name', 'cycle_type'])

def plot_current_product_volume_kr(current_product_kr):
    plot_df = current_product_kr.copy().sort_values(['product_name', 'cycle_type'])

    fig = make_subplots(specs=[[{'secondary_y': True}]])
    for cycle_type in ['Base', 'Extra']:
        d = plot_df[plot_df['cycle_type'].astype(str) == cycle_type]
        opacity = 1.0 if cycle_type == 'Base' else 0.45
        fig.add_bar(
            x=d['product_name'].astype(str),
            y=d['vol_original_window'],
            name=f'Volume Alocado — {cycle_type}',
            marker_color=colors_cycle.get(cycle_type, '#ad3365'),
            opacity=opacity,
            text=[f'{v:,.0f}' for v in d['vol_original_window']],
            textposition='outside',
            secondary_y=False,
        )
        fig.add_trace(
            go.Scatter(
                x=d['product_name'].astype(str),
                y=d['okr_weighted_pct'],
                name=f'OKR Atual — {cycle_type} (%)',
                mode='lines+markers+text',
                text=[f'{v:.0f}%' for v in d['okr_weighted_pct']],
                textposition='top center',
                line=dict(
                    color=colors_cycle.get(cycle_type, 'black'),
                    width=2,
                    dash='solid' if cycle_type == 'Base' else 'dash',
                    shape='spline',
                ),
                marker=dict(color=colors_cycle.get(cycle_type, 'black'), size=7),
            ),
            secondary_y=True,
        )

    fig.update_layout(
        title='Volume Alocado e OKR Atual por Produto — Base vs Extra'
              '<br><sup>Top 10 produtos por volume alocado na janela Cn..Cn+3; apenas alterações internas</sup>',
        template='plotly_white',
        xaxis_tickangle=-45,
        barmode='group',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
        margin=dict(t=110),
    )
    fig.update_yaxes(title_text='Volume Alocado (peças)', secondary_y=False)
    fig.update_yaxes(title_text='OKR Atual (%)', range=[0, 105], secondary_y=True)
    fig.show()



In [4]:
# Query principal: plano original vs estado atual com flags INT/EXT
df_plano = load_plano(sql_path)
df_plano.head()


/Users/insider/LA_Coding_Projects/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Plano vs Atual: 62,589 linhas (OP-SKU), 256 ciclos


,op_code,product_sku,cycle_name,supplier_id,supplier_name,product_name,product_color,product_size,is_finished_product_order,production_order_type,...,is_int_date,is_int_cancel,is_int_grade,is_int_any,is_ext_cancel,is_ext_date_rev,is_ext_any,delta_days_planned,delta_days_reviewed,delta_planned_qty
0,OP0450023,102010100104,00,45,JKM,Tech T-shirt Gola U Masculino,Preto,P,False,committed,...,False,False,False,False,False,False,False,0,0,0
1,OP0450023,102010100105,00,45,JKM,Tech T-shirt Gola U Masculino,Preto,M,False,committed,...,False,False,False,False,False,False,False,0,0,0
2,OP0450023,102010100106,00,45,JKM,Tech T-shirt Gola U Masculino,Preto,G,False,committed,...,False,False,False,False,False,False,False,0,0,0
3,OP0450023,102010100107,00,45,JKM,Tech T-shirt Gola U Masculino,Preto,GG,False,committed,...,False,False,False,False,False,False,False,0,0,0
4,OP0450023,102010100108,00,45,JKM,Tech T-shirt Gola U Masculino,Preto,XGG,False,committed,...,False,False,False,False,False,False,False,0,0,0


In [5]:
# Frequência de revisões por OP-SKU
df_freq = load_frequencia(sql_path)
df_freq.head()


/Users/insider/LA_Coding_Projects/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Frequência de revisões: 82,772 linhas (OP-SKU)


,op_code,product_sku,cycle_name,supplier_name,product_name,cycle_type,n_rev_planned,n_rev_reviewed_ext,n_rev_grade,n_rev_total,total_magnitude_dt_planned,total_magnitude_dt_reviewed,total_abs_delta_grade,first_revision_date,last_revision_date
0,OPF60N259,None,24-10-s4,DALOP,Action Top Feminino,Extra,0,0,1053,1053,0,0,77420,2025-11-10,2026-07-22
1,OPF61N9,None,24-07-m1,N8,NoHo Socks,Extra,0,0,380,380,0,0,46740,2025-11-10,2026-07-22
2,OPF61N13,None,24-09-m1,N8,NoHo Socks,Extra,0,0,369,369,0,0,151290,2025-11-10,2026-07-22
3,OPF61N10,None,24-07-m1,N8,NoHo Socks,Extra,0,0,367,367,0,0,45141,2025-11-10,2026-07-22
4,OPF61N20,None,24-10-s3,N8,NoHo Socks,Extra,0,0,365,365,0,0,365000,2025-11-10,2026-07-22


# 2. Cálculo do KR1 por Coorte

Para cada `cycle_name`, calcula:
- **volume_original**: soma de `baseline_planned_qty`
- **volume_alterado**: soma de `baseline_planned_qty` onde `is_int_any = True`
- **KR1** = 1 - (volume_alterado / volume_original)


In [6]:
df_plano, df_freq, kr1_coorte, ciclos_validos = build_kr1_coorte(
    df_plano,
    df_freq,
    CICLOS_EXCLUIDOS,
)


Ciclos excluídos: ['C012026']
df_plano após exclusão: 61,662 linhas, 255 ciclos
Coortes após filtro (mes_alvo >= 2025-11): 173
df_plano filtrado: 23,540 linhas
df_freq filtrado: 35,983 linhas

Total de coortes: 173
  Base: 15
  Extra: 158


/var/folders/m7/t8xk69wj7xl3lrprq0gflfqc0000gn/T/ipykernel_55596/3645247067.py:61: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mes_alvo = df_plano.groupby('cycle_name').apply(


In [7]:
kr1_coorte[['cycle_name', 'cycle_type', 'mes_alvo', 'vol_original', 'vol_int_any', 'kr1_pct', 'n_ops']]


,cycle_name,cycle_type,mes_alvo,vol_original,vol_int_any,kr1_pct,n_ops
73,C112025,Base,2025-11,709871,59674,91.6,397
165,EPA2706,Extra,2025-11,75678,0,100.0,49
215,LAN1509,Extra,2025-11,73561,0,100.0,22
236,NEXTECH,Extra,2025-11,40014,0,100.0,45
108,EPA08TPT,Extra,2025-11,29998,0,100.0,8
...,...,...,...,...,...,...,...
234,LAN3006,Extra,2026-11,38448,0,100.0,123
182,IMPORTACAO1706,Extra,2026-11,8003,8003,0.0,13
117,EPA1307,Extra,2026-11,7036,2878,59.1,7
232,LAN2906,Extra,2026-11,4160,0,100.0,22


# 3. KR1 Consolidado (KR1a + KR1b + Total)

- **KR1a** (ciclos base): meta ≥ 85%
- **KR1b** (extras): meta ≥ 70%
- **KR1 consolidado**: média ponderada por volume


In [8]:
kr1_tipo, kr1_total = build_kr1_tipo(kr1_coorte)


Coortes no OKR ativo (mes_alvo >= 2026-07): 42
=== KR1 por Tipo de Ciclo ===
cycle_type  vol_original  vol_int_any  kr1_pct  meta
      Base       1517863       319774     78.9  85.0
     Extra        384423        82160     78.6  70.0

=== KR1 Consolidado (OKR ativo): 78.9% ===
Volume original total: 1,902,286
Volume alterado (INT): 401,934


In [9]:
kr1_mes, kr1_mes_total = build_kr1_mes(kr1_coorte)


=== KR1 por Mês-Alvo ===
mes_alvo_str cycle_type  vol_original  kr1_pct  n_coortes
     2025-11       Base        709871     91.6          1
     2025-11      Extra        377645     96.9         18
     2025-12       Base        536059     94.6          1
     2025-12      Extra        218123     71.9         17
     2026-01       Base        396201     89.8          3
     2026-01      Extra        237276     58.4         14
     2026-02       Base        361086     81.7          1
     2026-02      Extra         91526     85.8         13
     2026-03       Base        299397     60.5          1
     2026-03      Extra        310910     76.5         19
     2026-04       Base        356484     24.1          1
     2026-04      Extra        147992     38.2         18
     2026-05       Base        282514     15.9          1
     2026-05      Extra         62899     26.8          8
     2026-06       Base        339083     12.7          1
     2026-06      Extra         44863     30.7 

# 4. Visualizações — Visão de Planejamento


In [10]:
plot_kr1_mes(kr1_mes, colors_cycle)


In [11]:
df_plano = add_waterfall_reason_code(df_plano)
df_waterfall = build_waterfall_data(df_plano, kr1_coorte, cycle_type=WATERFALL_CYCLE_TYPE)
plot_waterfall(df_waterfall, cycle_type=WATERFALL_CYCLE_TYPE, mes_escolhido=WATERFALL_MES_ESCOLHIDO)


In [12]:
df_rc, pct_total_int, vol_ext_mes, colors_rc = build_reason_code_data(kr1_mes)
plot_reason_code_breakdown(df_rc, pct_total_int, vol_ext_mes, colors_rc)


In [13]:
plot_kr1_coorte_top20(kr1_coorte, colors_cycle)


# 5. Análise de Frequência de Revisões


In [14]:
df_freq, freq_dist, pct_sem_rev, pct_3_mais = prepare_freq_distribution(df_freq)
plot_freq_distribution(freq_dist, colors_cycle)
print(f"\n{pct_sem_rev:.1f}% das OPs nunca tiveram revisão")
print(f"{pct_3_mais:.1f}% tiveram 3+ revisões")


/var/folders/m7/t8xk69wj7xl3lrprq0gflfqc0000gn/T/ipykernel_55596/3645247067.py:377: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  freq_dist = df_freq_op.groupby(['faixa_rev', 'cycle_type']).agg(



60.4% das OPs nunca tiveram revisão
21.3% tiveram 3+ revisões


In [15]:
heatmap_pct = build_supplier_heatmap(df_freq)
plot_supplier_heatmap(heatmap_pct)


/var/folders/m7/t8xk69wj7xl3lrprq0gflfqc0000gn/T/ipykernel_55596/3645247067.py:415: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  heatmap_data = df_freq_top.groupby(['supplier_name', 'faixa_rev_ext']).size().reset_index(name='count')


In [16]:
df_plano


,op_code,product_sku,cycle_name,supplier_id,supplier_name,product_name,product_color,product_size,is_finished_product_order,production_order_type,...,is_int_cancel,is_int_grade,is_int_any,is_ext_cancel,is_ext_date_rev,is_ext_any,delta_days_planned,delta_days_reviewed,delta_planned_qty,int_reason_excl
31764,OPF101N16,203120720104,Alocação extra Adapt Fit,101,CLARA BELLA,Stirrup Legging Feminino,Preto,P,True,committed,...,False,False,False,False,False,False,0,0,0,Inalterado
31765,OPF101N16,203120720105,Alocação extra Adapt Fit,101,CLARA BELLA,Stirrup Legging Feminino,Preto,M,True,committed,...,False,False,False,False,False,False,0,0,0,Inalterado
31766,OPF101N16,203120720106,Alocação extra Adapt Fit,101,CLARA BELLA,Stirrup Legging Feminino,Preto,G,True,committed,...,False,False,False,False,False,False,0,0,0,Inalterado
31767,OPF101N16,203120720107,Alocação extra Adapt Fit,101,CLARA BELLA,Stirrup Legging Feminino,Preto,GG,True,committed,...,False,False,False,False,False,False,0,0,0,Inalterado
31768,OPF101N17,203120720104,Alocação extra Adapt Fit,101,CLARA BELLA,Stirrup Legging Feminino,Preto,P,True,committed,...,False,False,False,False,False,False,0,0,0,Inalterado
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62562,OPF102N69,2095821010304,Whateverproof / LifeProof,102,BLUTEXTIL,HighTech T-shirt LifeProof Feminino,Azul Marinho,P,False,committed,...,False,False,False,True,False,True,0,0,0,Inalterado
62563,OPF102N69,2095821010305,Whateverproof / LifeProof,102,BLUTEXTIL,HighTech T-shirt LifeProof Feminino,Azul Marinho,M,False,committed,...,False,False,False,True,False,True,0,0,0,Inalterado
62564,OPF102N69,2095821010306,Whateverproof / LifeProof,102,BLUTEXTIL,HighTech T-shirt LifeProof Feminino,Azul Marinho,G,False,committed,...,False,False,False,True,False,True,0,0,0,Inalterado
62565,OPF102N69,2095821010307,Whateverproof / LifeProof,102,BLUTEXTIL,HighTech T-shirt LifeProof Feminino,Azul Marinho,GG,False,committed,...,False,False,False,True,False,True,0,0,0,Inalterado


In [17]:
df_freq, df_freq_rev, df_freq_ext, df_supplier_product_ext = prepare_instability_data(df_freq)
plot_instability_scatter(df_supplier_product_ext, colors_cycle)
print("\n=== Top 10 pares Supplier x Product_Name com mais OPs alteradas externamente ===")
print(df_supplier_product_ext.nlargest(10, 'n_ops_mudadas')[
    ['supplier_name', 'product_name', 'cycle_type', 'n_ops_mudadas', 'dias_deslocamento_medio', 'n_rev_ext_medio', 'total_rev_ext']
].to_string(index=False))



=== Top 10 pares Supplier x Product_Name com mais OPs alteradas externamente ===
supplier_name                  product_name cycle_type  n_ops_mudadas  dias_deslocamento_medio  n_rev_ext_medio  total_rev_ext
   BAE BRASIL      The Perfect Top Feminino       Base            102                      8.6             6.96            710
   BAE BRASIL Tech T-shirt Gola U Masculino       Base             81                     13.0              6.0            486
   BAE BRASIL       Daily T-shirt Masculino       Base             52                     20.0             6.67            347
   BAE BRASIL Tech T-shirt Gola U Masculino      Extra             50                      8.4             4.62            231
         DDAL             Wingsuit Feminino       Base             48                     10.3              3.9            187
    BY COTTON Tech T-shirt Gola U Masculino       Base             47                     10.2             2.47            116
   BAE BRASIL       Daily T-s

In [18]:
plot_instability_scatter_by_cycle(df_supplier_product_ext, 'Base')


In [19]:
plot_instability_scatter_by_cycle(df_supplier_product_ext, 'Extra')


In [20]:
df_treemap = build_external_treemap_data(df_supplier_product_ext)
plot_external_treemap(df_treemap)


# 6. Drill-Down: Timeline de uma OP específica

Selecione uma OP e SKU para ver a evolução de `dt_planned`, `dt_reviewed` e `planned_quantity` ao longo dos snapshots diários.


In [21]:
top_instavel, op_drill, sku_drill, df_timeline = load_timeline_drilldown(df_freq_rev, sql_path)
df_timeline.head(10)


Drill-down: OP OPF37N2057, SKU 102010109306
  Fornecedor: BAE BRASIL
  Ciclo: C062026
  Revisões: 40 (INT_DATE: 2, EXT_DATE_REV: 37, INT_GRADE: 1)


/Users/insider/LA_Coding_Projects/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,op_code,product_sku,cycle_name,supplier_name,product_name,ingestion_date,current_production_stage,planned_quantity,dt_planned_entry_warehouse,dt_reviewed_entry_warehouse,received_quantity,cutted_quantity,prev_dt_planned,prev_dt_reviewed,prev_planned_qty,change_dt_planned,change_dt_reviewed,change_grade
0,OPF37N2057,102010109306,C062026,BAE BRASIL,Tech T-shirt Gola U Masculino,2026-02-07,waiting_fabric_arrival,457,2026-06-01,2026-06-01,<NA>,<NA>,NaT,NaT,<NA>,None,None,None
1,OPF37N2057,102010109306,C062026,BAE BRASIL,Tech T-shirt Gola U Masculino,2026-02-08,waiting_fabric_arrival,457,2026-06-01,2026-06-01,<NA>,<NA>,2026-06-01,2026-06-01,457,None,None,None
2,OPF37N2057,102010109306,C062026,BAE BRASIL,Tech T-shirt Gola U Masculino,2026-02-09,waiting_fabric_arrival,457,2026-06-01,2026-06-01,<NA>,<NA>,2026-06-01,2026-06-01,457,None,None,None
3,OPF37N2057,102010109306,C062026,BAE BRASIL,Tech T-shirt Gola U Masculino,2026-02-10,waiting_fabric_arrival,457,2026-06-01,2026-06-01,<NA>,<NA>,2026-06-01,2026-06-01,457,None,None,None
4,OPF37N2057,102010109306,C062026,BAE BRASIL,Tech T-shirt Gola U Masculino,2026-02-11,waiting_fabric_arrival,457,2026-06-01,2026-06-01,<NA>,<NA>,2026-06-01,2026-06-01,457,None,None,None
5,OPF37N2057,102010109306,C062026,BAE BRASIL,Tech T-shirt Gola U Masculino,2026-02-12,waiting_fabric_arrival,457,2026-06-01,2026-06-01,<NA>,<NA>,2026-06-01,2026-06-01,457,None,None,None
6,OPF37N2057,102010109306,C062026,BAE BRASIL,Tech T-shirt Gola U Masculino,2026-02-13,waiting_fabric_arrival,457,2026-06-01,2026-06-01,<NA>,<NA>,2026-06-01,2026-06-01,457,None,None,None
7,OPF37N2057,102010109306,C062026,BAE BRASIL,Tech T-shirt Gola U Masculino,2026-02-14,waiting_fabric_arrival,457,2026-06-01,2026-06-01,<NA>,<NA>,2026-06-01,2026-06-01,457,None,None,None
8,OPF37N2057,102010109306,C062026,BAE BRASIL,Tech T-shirt Gola U Masculino,2026-02-15,waiting_fabric_arrival,457,2026-06-01,2026-06-01,<NA>,<NA>,2026-06-01,2026-06-01,457,None,None,None
9,OPF37N2057,102010109306,C062026,BAE BRASIL,Tech T-shirt Gola U Masculino,2026-02-16,waiting_fabric_arrival,457,2026-06-01,2026-06-01,<NA>,<NA>,2026-06-01,2026-06-01,457,None,None,None


In [22]:
df_timeline = prepare_timeline_data(df_timeline)
plot_timeline_datas(df_timeline, op_drill, sku_drill)


In [23]:
plot_timeline_grade(df_timeline, op_drill, sku_drill)


# 7. Evolução Temporal do KR1

Compara cada snapshot semanal contra o baseline para visualizar como o KR1 "degrada" ao longo do tempo.
Permite identificar **quando** as mudanças acontecem e quais meses-alvo são mais impactados.


In [24]:
df_evolucao = load_evolucao(sql_path, CICLOS_EXCLUIDOS, ciclos_validos, kr1_coorte)
df_evolucao.head()


/Users/insider/LA_Coding_Projects/.venv/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Evolução temporal: 93,680 linhas (semana × ciclo × fornecedor × produto)
Semanas: 37, Ciclos: 254, Fornecedores: 82, Produtos: 230

Após filtro: 28,227 linhas, 172 ciclos


,snapshot_week,cycle_name,cycle_type,supplier_name,product_name,vol_original,vol_int_date,vol_int_cancel,vol_int_grade,vol_int_any,vol_ext_cancel,vol_ext_date_rev,vol_ext_any,mes_alvo,kr1,kr1_pct,mes_alvo_str
0,2025-11-10,Alocação extra Adapt Fit,Extra,CLARA BELLA,Calça Flare InSkin Feminino,2000,0,0,0,0,0,0,0,2025-11,1.0,100.0,2025-11
1,2025-11-10,Alocação extra Adapt Fit,Extra,CLARA BELLA,Modern Top Feminino,1600,0,0,0,0,0,0,0,2025-11,1.0,100.0,2025-11
2,2025-11-10,Alocação extra Adapt Fit,Extra,CLARA BELLA,Stirrup Legging Feminino,12000,0,0,0,0,0,2000,2000,2025-11,1.0,100.0,2025-11
3,2025-11-10,Alocação extra Adapt Fit,Extra,CLARA BELLA,Zipper Legging Feminino,2000,0,0,0,0,0,250,250,2025-11,1.0,100.0,2025-11
4,2025-11-10,BOLD 2 - Revisada,Extra,WARUSKY,Boxy Cropped InLounge Feminino,6408,0,0,0,0,0,0,0,2026-01,1.0,100.0,2026-01


In [25]:
evol_mes = build_evol_mes(df_evolucao)
plot_evol_kr1_mes(evol_mes)


In [26]:
evol_tipo = build_evol_tipo(df_evolucao)
plot_evol_kr1_tipo(evol_tipo)


In [27]:
heatmap_pivot = build_evol_heatmap_base(evol_mes)
plot_evol_heatmap_base(heatmap_pivot)


# 7a. OKR Ponderado e Visões Internas Adicionais

Novas visões ancoradas exclusivamente nas alterações internas (`INT_CANCEL`, `INT_DATE`, `INT_GRADE`).

O OKR ponderado usa a janela móvel `Cn..Cn+3`, onde `Cn` é o mês do `snapshot_week`:

`OKR = (35 * taxa Cn + 35 * taxa Cn+1 + 20 * taxa Cn+2 + 10 * taxa Cn+3) / 100`


In [28]:
okr_evol_tipo = build_weighted_okr_evolution(df_evolucao, include_cycle_type=True)
plot_weighted_okr_evolution(okr_evol_tipo)


In [29]:
allocated_stack = build_allocated_volume_stack(df_plano, kr1_coorte)
plot_allocated_volume_stack(allocated_stack)


In [30]:
internal_change_bars, internal_change_line = build_internal_change_pct_by_month(kr1_mes)
plot_internal_change_pct_by_month(internal_change_bars, internal_change_line)


In [31]:
df_evolucao_base = df_evolucao[df_evolucao['cycle_type'] == 'Base'].copy()

okr_evol_supplier = build_weighted_okr_evolution(
    df_evolucao_base,
    group_cols=['supplier_name'],
    include_cycle_type=False,
)
plot_weighted_okr_by_dimension(
    okr_evol_supplier,
    'supplier_name',
    'Evolução do OKR Ponderado por Fornecedor — Ciclos Base',
    top_n=10,
)


In [32]:
df_evolucao_base = df_evolucao[df_evolucao['cycle_type'] == 'Base'].copy()

okr_evol_product = build_weighted_okr_evolution(
    df_evolucao_base,
    group_cols=['product_name'],
    include_cycle_type=False,
)
plot_weighted_okr_by_dimension(
    okr_evol_product,
    'product_name',
    'Evolução do OKR Ponderado por Produto — Ciclos Base',
    top_n=10,
)


In [33]:
current_supplier_okr = build_current_supplier_volume_okr(df_evolucao, top_n=10)
plot_current_supplier_volume_okr(current_supplier_okr)


In [34]:
current_product_kr = build_current_product_volume_kr(df_evolucao, top_n=10)
plot_current_product_volume_kr(current_product_kr)


# 7b. Evolução Temporal das Alterações Externas

Mesma lógica da seção anterior, mas olhando apenas alterações externas ao plano.
Aqui o foco é medir **% do volume original impactado por fatores externos** ao longo do tempo.


In [35]:
evol_ext_mes = build_evol_ext_mes(df_evolucao)
plot_evol_ext_mes(evol_ext_mes)


In [36]:
evol_ext_tipo = build_evol_ext_tipo(df_evolucao)
plot_evol_ext_tipo(evol_ext_tipo)


In [37]:
heatmap_ext_pivot = build_evol_ext_heatmap_base(evol_ext_mes)
plot_evol_ext_heatmap_base(heatmap_ext_pivot)


# 8. Export


In [38]:
export_outputs(output_path, today, kr1_coorte, kr1_mes, df_plano, df_freq, df_evolucao)


Arquivos exportados em ../../outputs/ com sufixo _20260722
